# fno-thermal-design: run all experiments on Colab

1. `Runtime -> Change runtime type -> T4 GPU`, then run the cells top to bottom.
2. The project folder lives on **Google Drive**, so results and checkpoints survive a disconnect.
3. **After a disconnect:** reconnect, run the *Mount Drive* cell, then re-run the experiment cell that was interrupted.
   Finished runs are skipped and unfinished ones resume from their last checkpoint.
4. `!python` cells do **not** stop on errors, so check the end of each cell's output for a `Traceback`.
5. Steps 01, 02, 07 and 09 do not need a GPU. If the GPU quota runs out, you can still download `results/` from drive.google.com.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo 'No GPU: switch the runtime to T4 GPU'

### Mount Drive (run this first after every reconnect)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
WORK = '/content/drive/MyDrive/fno-thermal-design'
print('project folder:', WORK, '| exists:', os.path.exists(WORK))

### Get the code (first time only)
Upload `fno-thermal-design.zip` when prompted, or clone the repository:
`!git clone https://github.com/dgh19981102-max/fno-thermal-design.git "$WORK"`

In [ ]:
import os, zipfile
if not os.path.exists(os.path.join(WORK, 'src')):
    from google.colab import files
    up = files.upload()                      # choose fno-thermal-design.zip
    with zipfile.ZipFile(next(iter(up))) as z:
        z.extractall('/content/drive/MyDrive/')
print(sorted(os.listdir(WORK)))

In [ ]:
%cd {WORK}
!pip install -q -r requirements.txt
!python -m pytest -q

## 01 Verify the solver (~1 min, CPU)

In [ ]:
%cd {WORK}/experiments
!python 01_verify_solver.py

## 02 Generate datasets (~2-3 min, CPU)

In [ ]:
%cd {WORK}/experiments
!python 02_generate_data.py

## 03 Benchmark: FNO / U-Net / DeepONet x 5 seeds (~1.5 h on a T4)

In [ ]:
%cd {WORK}/experiments
!python 03_train_benchmark.py

## 04 Physics-loss ablation, 27 runs (~3 h on a T4)

In [ ]:
%cd {WORK}/experiments
!python 04_physics_ablation.py

## 05 Inference speed (~2 min)

In [ ]:
%cd {WORK}/experiments
!python 05_speed_benchmark.py

## 06 Design optimisation: solver vs surrogate (~6 min)

In [ ]:
%cd {WORK}/experiments
!python 06_design_optimization.py

## 08 FNO generalisation diagnostics, 9 runs (~1 h on a T4)

In [ ]:
%cd {WORK}/experiments
!python 08_fno_diagnostics.py

## 07 Tables and figures, 09 cover image (CPU)

In [ ]:
%cd {WORK}/experiments
!python 07_make_figures.py
!python 09_make_cover.py
from IPython.display import Image, display
import glob
for p in sorted(glob.glob(f'{WORK}/docs/figures/*.png')) + [f'{WORK}/docs/upwork_cover.png']:
    print(p); display(Image(p, width=900))

## Download results

In [ ]:
%cd {WORK}
!rm -f fno_results.zip && zip -qr fno_results.zip results docs/figures docs/upwork_cover.png
from google.colab import files
files.download('fno_results.zip')